# Driftvarsling med OpenDrift

I simulatorene i boka har du sett hvordan strøm, vind og tilfeldig blanding flytter ting på havet. I denne notatboka bruker du det samme verktøyet som Meteorologisk institutt bruker når de varsler oljesøl og hjelper redningstjenesten, nemlig **OpenDrift**. Strømmen kommer fra havmodellen **NorKyst** og vinden fra værmodellen **MEPS**, begge med ferske varsler fra MET.

Du skal

1. kjøre en enkel drift med konstant strøm og vind og kjenne igjen vindfaktoren fra simulatoren,
2. hente et ekte varsel og se hvor noe driver utenfor kysten de neste døgnene,
3. sammenligne en person i vannet med en redningsflåte og regne bakover i tid,
4. se hvordan ulike oljetyper forvitrer,
5. se hvordan strømmen endrer seg med dypet.

Celler med ✏️ har spørsmål du skal svare på. Skriv svarene i en ny tekstcelle under spørsmålet.

## Før du starter

Du trenger Miniforge og et eget conda-miljø som heter `opendrift`. Oppskriften *Installere Python og OpenDrift* viser hvordan, både for Windows og Mac. Har du alt Miniforge fra matematikk, holder det å kjøre denne kommandoen i Miniforge Prompt (Windows) eller Terminal (Mac).

```
conda create -n opendrift -c conda-forge python=3.12 opendrift jupyterlab cmocean
```

Start notatboka med `conda activate opendrift` og deretter `jupyter lab`.

Notatboka henter data fra `thredds.met.no`, så du må være koblet til internett. Hvis MET-serveren ikke svarer, bruker notatboka konstante felt i stedet, og det står i utskriften.

In [ ]:
import sys
print("Python fra", sys.executable)   # skal ende på .../envs/opendrift/bin/python

from datetime import datetime, timedelta, timezone
import numpy as np
import matplotlib.pyplot as plt

from opendrift.models.oceandrift import OceanDrift
from opendrift.models.leeway import Leeway
from opendrift.models.openoil import OpenOil
from opendrift.readers import reader_netCDF_CF_generic

# Fargene fra Jordens puls
JP = {"havbla": "#17566D", "brent_oransje": "#C65F37", "bolgeturkis": "#57979E",
      "rav": "#F3A348", "disbla": "#7798B4", "midnattsbla": "#10283F"}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=list(JP.values())),
                     "axes.formatter.use_locale": False})

## 1 Drift med konstant strøm og vind

Vi starter uten ekte data. Strømmen er 0,1 m/s mot nord, og vinden blåser fra vest med 10 m/s. Vi slipper ut 300 partikler på samme sted. En tredel får vindfaktor 0 %, en tredel 1 % og en tredel 3 %, akkurat som valgene i simulatoren *Drift med strøm og vind*.

I OpenDrift er vinden og strømmen gitt som komponenter. `x` er mot øst og `y` er mot nord, og retningen er den vinden og strømmen går **mot**.

In [ ]:
o = OceanDrift(loglevel=50)   # loglevel=50 skjuler de fleste meldinger
o.set_config("environment:constant:x_sea_water_velocity", 0.0)   # m/s mot øst
o.set_config("environment:constant:y_sea_water_velocity", 0.1)   # m/s mot nord
o.set_config("environment:constant:x_wind", 10.0)                # vind fra vest = vind mot øst
o.set_config("environment:constant:y_wind", 0.0)
o.set_config("environment:constant:land_binary_mask", 0)         # bare hav
o.set_config("general:use_auto_landmask", False)
o.set_config("environment:constant:horizontal_diffusivity", 5)                  # m²/s, tilfeldig blanding

vindfaktor = np.repeat([0.0, 0.01, 0.03], 100)
start = datetime(2026, 1, 1)
o.seed_elements(lon=3.0, lat=60.0, radius=200, number=300, time=start,
                wind_drift_factor=vindfaktor)
o.run(duration=timedelta(hours=24), time_step=900, time_step_output=3600)

o.plot(fast=True, linecolor="wind_drift_factor", cmap="viridis",
       title="Drift i 24 timer, farget etter vindfaktor");

In [ ]:
# Hvor langt mot øst har hver gruppe flyttet seg etter 24 timer?
lon = o.result.lon.isel(time=-1).values
lat = o.result.lat.isel(time=-1).values
dx_km = (lon - 3.0) * 111.32 * np.cos(np.radians(60)) # km per lengdegrad ved 60° N
for f in [0.0, 0.01, 0.03]:
    utvalg = vindfaktor == f
    print(f"Vindfaktor {f*100:.0f} %: {dx_km[utvalg].mean():5.1f} km mot øst")

✏️ **Spørsmål 1**

a) Regn ut for hånd hvor langt en gjenstand med vindfaktor 3 % flytter seg mot øst på 24 timer når vinden er 10 m/s. Stemmer det med utskriften?

b) Hvorfor flytter gruppene seg like langt mot nord?

## 2 Et ekte varsel utenfor Vestlandskysten

Nå henter vi ferske varsler fra MET. **NorKyst** er en havmodell med 800 m oppløsning langs hele norskekysten. **MEPS** er værmodellen MET bruker i de vanlige værvarslene, med 2,5 km oppløsning. Begge oppdateres flere ganger i døgnet, så resultatet blir ulikt hver gang du kjører notatboka.

Funksjonen nedenfor kobler til dataene. Det kan ta et halvt minutt.

In [ ]:
NORKYST = "https://thredds.met.no/thredds/dodsC/fou-hi/norkystv3_800m_m00_be"
MEPS = "https://thredds.met.no/thredds/dodsC/mepslatest/meps_lagged_6_h_latest_2_5km_latest.nc"

def hent_varsel():
    # Gir lesere for strøm og vind, starttid og hvor mange timer varselet holder.
    try:
        hav = reader_netCDF_CF_generic.Reader(NORKYST)
        vaer = reader_netCDF_CF_generic.Reader(MEPS)
    except Exception as feil:
        print("Fikk ikke kontakt med MET. Bruker konstante felt i stedet.\n", feil)
        return None, datetime(2026, 1, 1), 48
    naa = datetime.now(timezone.utc).replace(tzinfo=None, minute=0, second=0, microsecond=0)
    start = max(naa, vaer.start_time, hav.start_time)
    slutt = min(vaer.end_time, hav.end_time)
    timer = int(min(48, (slutt - start).total_seconds() / 3600))
    print("Havmodell:", hav.name, "\nVærmodell:", vaer.name)
    print(f"Starttid {start:%d.%m.%Y kl. %H} UTC, varselet holder i {timer} timer")
    return [hav, vaer], start, timer

def legg_til_varsel(o, lesere):
    if lesere is None:   # reserve uten nett
        o.set_config("environment:constant:y_sea_water_velocity", 0.1)
        o.set_config("environment:constant:x_wind", 7.0)
        o.set_config("environment:constant:y_wind", 7.0)
    else:
        o.add_reader(lesere)

lesere, START, TIMER = hent_varsel()

Vi slipper ut drivende partikler like vest for Øygarden. Prøv gjerne å endre posisjonen. Lengdegrad (`lon`) øker mot øst, og breddegrad (`lat`) øker mot nord. Hvis du velger et punkt på land, sier OpenDrift fra.

In [ ]:
LON, LAT = 4.70, 60.50   # vest for Øygarden

o = OceanDrift(loglevel=50)
legg_til_varsel(o, lesere)
o.set_config("environment:constant:horizontal_diffusivity", 10)
o.seed_elements(lon=LON, lat=LAT, radius=500, number=500, time=START,
                wind_drift_factor=0.03)
o.run(duration=timedelta(hours=TIMER), time_step=900, time_step_output=3600)

o.plot(fast=True, background=["x_sea_water_velocity", "y_sea_water_velocity"],
       cmap="Blues", vmin=0, vmax=0.8, clabel="Strømfart (m/s)",
       title=f"Drift med 3 % vindfaktor fra {START:%d.%m. kl. %H} UTC");

In [ ]:
# Hvor mange partikler har strandet?
nr = o.status_categories.index("stranded")
strandet = int((o.result.status == nr).any(dim="time").sum())
print(f"{strandet} av {o.result.sizes['trajectory']} partikler har strandet etter {TIMER} timer")

✏️ **Spørsmål 2**

a) Hvilken vei driver partiklene? Sjekk vindvarselet for området på yr.no. Passer driften med vindretningen og med strømmen i bakgrunnen av kartet?

b) Hvorfor sprer partiklene seg mer i dette varselet enn i del 1, selv om den tilfeldige blandingen er omtrent den samme?

c) Kjør notatboka igjen i morgen. Hvorfor blir resultatet annerledes?

## 3 Søk og redning

Redningstjenesten bruker modellen **Leeway**. Den har målte driftsegenskaper for over 80 typer gjenstander, fra personer i vannet til redningsflåter og småbåter. Vi sammenligner en person i vannet (`PIW-1`, driver med omtrent 1 % av vindfarten) og en redningsflåte uten ballast (`LIFE-RAFT-NB-1`, omtrent 4 %).

In [ ]:
kjoringer = []
for objekt in [1, 26]:          # 1 = person i vannet, 26 = redningsflåte uten ballast
    o = Leeway(loglevel=50)
    legg_til_varsel(o, lesere)
    o.seed_elements(lon=LON, lat=LAT, radius=300, number=300, time=START, object_type=objekt)
    o.run(duration=timedelta(hours=min(TIMER, 24)), time_step=900, time_step_output=3600)
    kjoringer.append(o)

# Tegn sporene til personene, og legg sluttposisjonene til begge på kartet
person, flate = kjoringer
ax, fig = person.plot(fast=True, show_elements=False, show_initial=False, show=False,
                      buffer=0.3, title="Hvor er de etter et døgn?")
for o, navn, farge, merke in [(person, "Person i vannet", JP["havbla"], "o"),
                              (flate, "Redningsflåte", JP["brent_oransje"], "^")]:
    lon = o.result.lon.ffill("time").isel(time=-1)
    lat = o.result.lat.ffill("time").isel(time=-1)
    ax.scatter(lon, lat, s=14, c=farge, marker=merke, label=navn,
               transform=person.crs_lonlat, zorder=10)
    print(f"{navn}: midtpunkt {float(lon.mean()):.2f}° Ø, {float(lat.mean()):.2f}° N")
ax.scatter(LON, LAT, s=80, c="k", marker="*", label="Startpunkt", transform=person.crs_lonlat, zorder=11)
ax.legend(loc="upper left")
plt.show()

### Bakover i tid

En tom redningsflåte er funnet. Hvor kan den ha kommet fra? Vi starter der den ble funnet og regner bakover ved å gi et negativt tidssteg. Flåten driver med omtrent 3,7 % av vindfarten, og vi bruker en vanlig OceanDrift-modell med den vindfaktoren.

In [ ]:
FUNNET_LON, FUNNET_LAT = 4.50, 60.95      # endre til et punkt i havet du vil teste
funnet_tid = START + timedelta(hours=min(TIMER, 24))

o = OceanDrift(loglevel=50)
legg_til_varsel(o, lesere)
o.set_config("environment:constant:horizontal_diffusivity", 10)
o.seed_elements(lon=FUNNET_LON, lat=FUNNET_LAT, radius=200, number=500,
                time=funnet_tid, wind_drift_factor=0.037)
o.run(duration=timedelta(hours=min(TIMER, 24)), time_step=-900, time_step_output=3600)

o.plot(fast=True, title="Bakover i tid fra funnstedet");

✏️ **Spørsmål 3**

a) Hvor langt fra hverandre er personen og flåten etter et døgn? Hva betyr det for hvor redningsmannskapet bør lete?

b) Hvorfor blir området der flåten kan ha kommet fra, større jo lenger tilbake i tid modellen regner?

c) Noen partikler i bakoverkjøringen havner på land. Hva betyr det?

## 4 Oljesøl og forvitring

OpenOil bruker målte egenskaper for oljetyper fra norske felt. Vi sammenligner tre oljer i 24 timer med jevn vind på 10 m/s og uten strøm, slik at vi bare ser på forvitringen. Svevende dråper kan komme opp igjen når vinden løyer, mens olje som er spredt for godt, er blandet så fint ut at den ikke kommer tilbake. Sammenlign med simulatoren *Oljesøl, drift og forvitring* i boka.

In [ ]:
oljer = {"GENERIC DIESEL": "Diesel",
         "TROLL, STATOIL": "Råolje fra Troll",
         "IFO-380LS 2014": "Tung bunkersolje (IFO-380)"}

KAT = [("mass_surface", "På overflaten", JP["brent_oransje"], None),
       ("mass_evaporated", "Fordampet", JP["disbla"], "//"),
       ("mass_submerged", "Svevende dråper i vannet", JP["havbla"], ".."),
       ("mass_dispersed", "Spredt for godt i vannet", JP["bolgeturkis"], "xx"),
       ("mass_biodegraded", "Brutt ned av bakterier", JP["rav"], "\\\\"),
       ("mass_stranded", "Strandet", JP["midnattsbla"], "++")]

fig, akser = plt.subplots(1, 3, figsize=(13, 4), sharey=True, constrained_layout=True)
for ax, (navn, norsk) in zip(akser, oljer.items()):
    o = OpenOil(loglevel=50, location="NORWAY")
    for k, v in {"x_wind": 10, "y_wind": 0, "x_sea_water_velocity": 0,
                 "y_sea_water_velocity": 0, "land_binary_mask": 0}.items():
        o.set_config(f"environment:constant:{k}", v)
    o.set_config("general:use_auto_landmask", False)
    o.seed_elements(lon=4.0, lat=60.0, time=datetime(2026, 1, 1), number=500, oil_type=navn)
    o.run(duration=timedelta(hours=24), time_step=900)
    b = o.get_oil_budget()
    timer = (o.result.time - o.result.time[0]).dt.total_seconds() / 3600
    lag = [100 * b[k] / b["mass_total"] for k, *_ in KAT]
    poly = ax.stackplot(timer, *lag, colors=[f for *_, f, _ in KAT], labels=[n for _, n, *_ in KAT])
    for p, (*_, monster) in zip(poly, KAT):
        p.set_hatch(monster); p.set_edgecolor("white")
    ax.set_title(norsk); ax.set_xlabel("Tid (timer)"); ax.set_xlim(0, 24); ax.set_ylim(0, 100)

akser[0].set_ylabel("Andel av oljen (%)")
handles, labels = akser[0].get_legend_handles_labels()
fig.legend(handles[::-1], labels[::-1], loc="outside right center")
fig.suptitle("Oljebudsjett ved vind 10 m/s")
plt.show()

Nå slipper vi ut råolje fra Troll vest for Øygarden med det ekte varselet. OpenOil lager et oljebudsjett for hele utslippet. Figuren har engelske ord.

| Engelsk | Norsk |
|---|---|
| surface | på overflaten |
| evaporated | fordampet |
| submerged | svevende dråper i vannet |
| dispersed | spredt for godt i vannet |
| stranded | strandet |
| emulsion viscosity | hvor seig emulsjonen er |
| water content | andel vann i emulsjonen |
| wind speed, current speed | vindfart, strømfart |

In [ ]:
o = OpenOil(loglevel=50, location="NORWAY")
legg_til_varsel(o, lesere)
o.seed_elements(lon=LON, lat=LAT, radius=500, number=500, time=START,
                oil_type="TROLL, STATOIL", z=0)
o.run(duration=timedelta(hours=TIMER), time_step=900, time_step_output=3600)
o.plot(fast=True, title="Råolje fra Troll");
o.plot_oil_budget(show_wind_and_current=True);

✏️ **Spørsmål 4**

a) Hvilken olje fordamper mest, og hvorfor?

b) Hva skjer med diesel i sterk vind? Hvorfor er det vanskeligere å samle opp et dieselutslipp med lenser enn et utslipp av tung bunkersolje?

c) Ved virkelige utslipp blir tung bunkersolje ofte liggende lenge på overflaten som en seig emulsjon av olje og vann. Sammenlign med resultatet for IFO-380 her og med simulatoren i boka. Hva kan være grunnen til forskjellen, og hva lærer det deg om å bruke modeller?

d) Se på oljebudsjettet fra det ekte varselet. Hvilke perioder har mest nedblanding? Sammenlign med vindstyrken nederst i figuren.

## 5 Strømmen i ulike dyp

NorKyst regner ut strømmen i mange dyp. Her slipper vi partikler jevnt fordelt fra overflaten ned til 50 m og farger sporene etter dypet. Vindfaktoren er 0, så partiklene følger bare strømmen. Sammenlign med simulatoren *Drift i ulike dyp og ekmanspiralen*.

In [ ]:
o = OceanDrift(loglevel=50)
legg_til_varsel(o, lesere)
o.set_config("drift:vertical_mixing", False)     # ingen turbulent blanding opp og ned
o.set_config("drift:vertical_advection", False)  # ingen vertikal strøm, partiklene holder dypet
o.set_config("environment:constant:horizontal_diffusivity", 5)
dyp = -np.random.default_rng(1).uniform(0, 50, 1000)   # negativ z er under overflaten
o.seed_elements(lon=LON, lat=LAT, z=dyp, radius=0, number=1000,
                time=START, wind_drift_factor=0)
o.run(duration=timedelta(hours=min(TIMER, 24)), time_step=900, time_step_output=3600)

o.plot(fast=True, linecolor="z", cmap="viridis_r", show_elements=False,
       vmin=-50, vmax=0, title="Drift i ulike dyp, farget etter z (m)");

✏️ **Spørsmål 5**

a) Driver partiklene i overflaten og på 50 m dyp i samme retning? Beskriv forskjellen.

b) Nær kysten er det ofte et tynt lag med ferskere vann øverst (Den norske kyststrømmen). Hvordan kan det påvirke strømmen i ulike dyp?

## Til slutt

✏️ Skriv et kort avsnitt om hva en driftsmodell trenger for å gi et godt varsel, og hvilke usikkerheter som er størst. Bruk eksempler fra notatboka.

**Kilder**

- Dagestad, K.-F., Röhrs, J., Breivik, Ø. og Ådlandsvik, B. (2018). OpenDrift v1.0: a generic framework for trajectory modelling. *Geoscientific Model Development*, 11, 1405–1420.
- Breivik, Ø. og Allen, A. A. (2008). An operational search and rescue model for the Norwegian Sea and the North Sea. *Journal of Marine Systems*, 69, 99–113.
- OpenDrift, <https://opendrift.github.io>. Data fra Meteorologisk institutt, <https://thredds.met.no>.

*Notatbok av Elisabeth Engum (KI), 2026.*